In [2]:
import json
import sys
from pathlib import Path
import importlib
import pandas as pd


In [3]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/phoothwincho/Desktop/Movie Knowledge Assistant


In [4]:
import src.retrieval as retrieval

from src.retrieval import (
    search_movies,
    hybrid_search_movies
)

from src.evaluation import evaluate_retrieval

print("Retrieval module:")
print(retrieval.__file__)

/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Retrieval module:
/Users/phoothwincho/Desktop/Movie Knowledge Assistant/notebook/../src/retrieval.py


In [5]:

from src.reranking import rerank_movies

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [7]:
#load ground truth
GROUND_TRUTH_PATH = (
    PROJECT_ROOT
    / "data"
    / "retrieval_ground_truth.json"
)

with open(
    GROUND_TRUTH_PATH,
    "r"
) as f:
    ground_truth = json.load(f)

print(
    f"Number of evaluation questions: "
    f"{len(ground_truth)}"
)

Number of evaluation questions: 30


In [8]:
#inspect ground truth
for i, item in enumerate(ground_truth[:5], start=1):
    print(f"\n{i}. Question:")
    print(item["question"])
    print("Relevant movies:")
    print(item["relevant_movies"])


1. Question:
A science fiction movie about space exploration
Relevant movies:
['Interstellar', 'The Martian']

2. Question:
A mind-bending science fiction movie
Relevant movies:
['Inception', 'Tenet']

3. Question:
A movie about time travel
Relevant movies:
['Back to the Future', 'Looper']

4. Question:
A superhero movie with Marvel characters
Relevant movies:
['Avengers: Endgame', 'Iron Man']

5. Question:
A superhero movie from DC
Relevant movies:
['The Dark Knight', 'Man of Steel']


In [9]:
#Test Dense Search

query = ground_truth[0]["question"]

vector_results = search_movies(
    query,
    limit=5
)

print("Question:")
print(query)

print("\nVector Search:")

for rank, movie in enumerate(
    vector_results,
    start=1
):
    print(
        f"{rank}. {movie['title']}"
    )

Question:
A science fiction movie about space exploration

Vector Search:
1. Red Planet
2. 3022
3. Prometheus
4. Sphere
5. Life


In [10]:
#hybrid search
hybrid_results = hybrid_search_movies(
    query,
    limit=5
)

print("Question:")
print(query)

print("\nHybrid Search:")

for rank, movie in enumerate(
    hybrid_results,
    start=1
):
    print(
        f"{rank}. {movie['title']}"
    )

Question:
A science fiction movie about space exploration

Hybrid Search:
1. Interstellar: Nolan's Odyssey
2. The Midnight Sky
3. Passengers
4. Ad Astra
5. Orbiter 9


In [11]:
#evaluate vector search
vector_report, vector_metrics = evaluate_retrieval(
    ground_truth=ground_truth,
    search_function=search_movies,
    top_k=5
)

print("Vector Search Results")
print("-" * 30)

for metric, score in vector_metrics.items():
    print(
        f"{metric}: {score:.4f}"
    )

Evaluating Retrieval: 100%|██████████| 30/30 [00:01<00:00, 20.09it/s]

Vector Search Results
------------------------------
Hit Rate: 0.1667
MRR: 0.0700


In [12]:
vector_report.head(10)

,question,expected,retrieved,hit,reciprocal_rank
0,A science fiction movie about space exploration,"Interstellar, The Martian","Red Planet, 3022, Prometheus, Sphere, Life",0,0.0
1,A mind-bending science fiction movie,"Inception, Tenet","Transcendence, Bliss, More, Multiverse, Parallel",0,0.0
2,A movie about time travel,"Back to the Future, Looper","Time Lapse, Timecrimes, Frequently Asked Quest...",0,0.0
3,A superhero movie with Marvel characters,"Avengers: Endgame, Iron Man","Marvel Studios: Assembling a Universe, Guardia...",0,0.0
4,A superhero movie from DC,"The Dark Knight, Man of Steel","DC Showcase: Death, Justice League: Gods and M...",0,0.0
5,A fantasy movie with magic,"Harry Potter and the Sorcerer's Stone, Fantast...","Magic Magic, Strange Magic, Bright, Magic in t...",0,0.0
6,A movie about dinosaurs,"Jurassic Park, Jurassic World","Dinosaur, The Good Dinosaur, The Dinosaur Proj...",0,0.0
7,A movie about robots and artificial intelligence,"Ex Machina, I, Robot","Automata, I, Robot, A.I. Artificial Intelligen...",1,0.5
8,A movie about survival in space,"Gravity, The Martian","3022, Life, Lost in Space, Attraction, Love",0,0.0
9,A movie about dreams,Inception,"The Science of Sleep, The Dreamers, On Body an...",0,0.0


In [13]:
#evaluate hybrid search
hybrid_report, hybrid_metrics = evaluate_retrieval(
    ground_truth=ground_truth,
    search_function=hybrid_search_movies,
    top_k=5
)

print("Hybrid Search Results")
print("-" * 30)

for metric, score in hybrid_metrics.items():
    print(
        f"{metric}: {score:.4f}"
    )

Evaluating Retrieval: 100%|██████████| 30/30 [00:01<00:00, 15.33it/s]

Hybrid Search Results
------------------------------
Hit Rate: 0.2000
MRR: 0.1022


In [14]:
hybrid_report.head(10)

,question,expected,retrieved,hit,reciprocal_rank
0,A science fiction movie about space exploration,"Interstellar, The Martian","Interstellar: Nolan's Odyssey, The Midnight Sk...",0,0.0
1,A mind-bending science fiction movie,"Inception, Tenet","Bliss, Transcendence, Metropia, Coherence, Adv...",0,0.0
2,A movie about time travel,"Back to the Future, Looper","Frequently Asked Questions About Time Travel, ...",0,0.0
3,A superhero movie with Marvel characters,"Avengers: Endgame, Iron Man","Marvel Studios: Assembling a Universe, Marvel ...",0,0.0
4,A superhero movie from DC,"The Dark Knight, Man of Steel",Lego Batman: The Movie - DC Super Heroes Unite...,0,0.0
5,A fantasy movie with magic,"Harry Potter and the Sorcerer's Stone, Fantast...","Onward, Wizards of Waverly Place: The Movie, U...",0,0.0
6,A movie about dinosaurs,"Jurassic Park, Jurassic World","Walking with Dinosaurs, The Dinosaur Project, ...",0,0.0
7,A movie about robots and artificial intelligence,"Ex Machina, I, Robot","I, Robot, A.I. Artificial Intelligence, Bigbug...",1,1.0
8,A movie about survival in space,"Gravity, The Martian","High Life, Passengers, Lost in Space, Prospect...",0,0.0
9,A movie about dreams,Inception,"In My Dreams, In Dreams, A House of Your Dream...",0,0.0


In [15]:
#compare 2 method
comparison = pd.DataFrame([
    {
        "Method": "Vector Search",
        "Hit Rate": vector_metrics["Hit Rate"],
        "MRR": vector_metrics["MRR"]
    },
    {
        "Method": "Hybrid Search",
        "Hit Rate": hybrid_metrics["Hit Rate"],
        "MRR": hybrid_metrics["MRR"]
    }
])

comparison

,Method,Hit Rate,MRR
0,Vector Search,0.166667,0.070000
1,Hybrid Search,0.200000,0.102222


In [16]:
#calculate improvement
hit_rate_difference = (
    hybrid_metrics["Hit Rate"]
    - vector_metrics["Hit Rate"]
)

mrr_difference = (
    hybrid_metrics["MRR"]
    - vector_metrics["MRR"]
)

print(
    f"Hit Rate difference: "
    f"{hit_rate_difference:+.4f}"
)

print(
    f"MRR difference: "
    f"{mrr_difference:+.4f}"
)

Hit Rate difference: +0.0333
MRR difference: +0.0322


In [17]:
#add percentage improvement
if vector_metrics["Hit Rate"] != 0:

    hit_rate_improvement = (
        (
            hybrid_metrics["Hit Rate"]
            - vector_metrics["Hit Rate"]
        )
        / vector_metrics["Hit Rate"]
    ) * 100

else:
    hit_rate_improvement = 0


if vector_metrics["MRR"] != 0:

    mrr_improvement = (
        (
            hybrid_metrics["MRR"]
            - vector_metrics["MRR"]
        )
        / vector_metrics["MRR"]
    ) * 100

else:
    mrr_improvement = 0


print(
    f"Hit Rate improvement: "
    f"{hit_rate_improvement:+.2f}%"
)

print(
    f"MRR improvement: "
    f"{mrr_improvement:+.2f}%"
)

Hit Rate improvement: +20.00%
MRR improvement: +46.03%


In [18]:
#Compare every query
comparison_details = []

for i, item in enumerate(
    ground_truth
):

    vector_row = vector_report.iloc[i]
    hybrid_row = hybrid_report.iloc[i]

    vector_rr = vector_row[
        "reciprocal_rank"
    ]

    hybrid_rr = hybrid_row[
        "reciprocal_rank"
    ]

    vector_hit = vector_row[
        "hit"
    ]

    hybrid_hit = hybrid_row[
        "hit"
    ]

    if hybrid_rr > vector_rr:
        winner = "Hybrid"

    elif vector_rr > hybrid_rr:
        winner = "Vector"

    elif hybrid_hit > vector_hit:
        winner = "Hybrid"

    elif vector_hit > hybrid_hit:
        winner = "Vector"

    else:
        winner = "Tie"

    comparison_details.append({
        "question": item["question"],

        "expected": ", ".join(
            item["relevant_movies"]
        ),

        "vector_retrieved": (
            vector_row["retrieved"]
        ),

        "vector_hit": vector_hit,

        "vector_rr": vector_rr,

        "hybrid_retrieved": (
            hybrid_row["retrieved"]
        ),

        "hybrid_hit": hybrid_hit,

        "hybrid_rr": hybrid_rr,

        "winner": winner
    })


comparison_details_df = pd.DataFrame(
    comparison_details
)

comparison_details_df

,question,expected,vector_retrieved,vector_hit,vector_rr,hybrid_retrieved,hybrid_hit,hybrid_rr,winner
0,A science fiction movie about space exploration,"Interstellar, The Martian","Red Planet, 3022, Prometheus, Sphere, Life",0,0.0,"Interstellar: Nolan's Odyssey, The Midnight Sk...",0,0.000000,Tie
1,A mind-bending science fiction movie,"Inception, Tenet","Transcendence, Bliss, More, Multiverse, Parallel",0,0.0,"Bliss, Transcendence, Metropia, Coherence, Adv...",0,0.000000,Tie
2,A movie about time travel,"Back to the Future, Looper","Time Lapse, Timecrimes, Frequently Asked Quest...",0,0.0,"Frequently Asked Questions About Time Travel, ...",0,0.000000,Tie
3,A superhero movie with Marvel characters,"Avengers: Endgame, Iron Man","Marvel Studios: Assembling a Universe, Guardia...",0,0.0,"Marvel Studios: Assembling a Universe, Marvel ...",0,0.000000,Tie
4,A superhero movie from DC,"The Dark Knight, Man of Steel","DC Showcase: Death, Justice League: Gods and M...",0,0.0,Lego Batman: The Movie - DC Super Heroes Unite...,0,0.000000,Tie
5,A fantasy movie with magic,"Harry Potter and the Sorcerer's Stone, Fantast...","Magic Magic, Strange Magic, Bright, Magic in t...",0,0.0,"Onward, Wizards of Waverly Place: The Movie, U...",0,0.000000,Tie
6,A movie about dinosaurs,"Jurassic Park, Jurassic World","Dinosaur, The Good Dinosaur, The Dinosaur Proj...",0,0.0,"Walking with Dinosaurs, The Dinosaur Project, ...",0,0.000000,Tie
7,A movie about robots and artificial intelligence,"Ex Machina, I, Robot","Automata, I, Robot, A.I. Artificial Intelligen...",1,0.5,"I, Robot, A.I. Artificial Intelligence, Bigbug...",1,1.000000,Hybrid
8,A movie about survival in space,"Gravity, The Martian","3022, Life, Lost in Space, Attraction, Love",0,0.0,"High Life, Passengers, Lost in Space, Prospect...",0,0.000000,Tie
9,A movie about dreams,Inception,"The Science of Sleep, The Dreamers, On Body an...",0,0.0,"In My Dreams, In Dreams, A House of Your Dream...",0,0.000000,Tie


In [19]:
#find where hybrid is improve
hybrid_better = comparison_details_df[
    (
        comparison_details_df["hybrid_rr"]
        >
        comparison_details_df["vector_rr"]
    )
]

print(
    "Queries where Hybrid has better MRR:"
)

print(
    len(hybrid_better)
)

hybrid_better

Queries where Hybrid has better MRR:
5


,question,expected,vector_retrieved,vector_hit,vector_rr,hybrid_retrieved,hybrid_hit,hybrid_rr,winner
7,A movie about robots and artificial intelligence,"Ex Machina, I, Robot","Automata, I, Robot, A.I. Artificial Intelligen...",1,0.5,"I, Robot, A.I. Artificial Intelligence, Bigbug...",1,1.000000,Hybrid
16,A movie directed by Christopher Nolan,"Interstellar, Inception, The Prestige, The Dar...","Interstellar: Nolan's Odyssey, Following, Dunk...",1,0.2,"Interstellar: Nolan's Odyssey, Following, Ince...",1,0.333333,Hybrid
17,A movie directed by Quentin Tarantino,"Pulp Fiction, Django Unchained","QT8: The First Eight, Grindhouse, Reality, Spi...",1,0.2,"QT8: The First Eight, Grindhouse, Django Uncha...",1,0.333333,Hybrid
19,A movie starring Tom Hanks,"Forrest Gump, Cast Away","I Saw the Light, Swiss Army Man, Greyhound, La...",0,0.0,"Greyhound, Larry Crowne, The Post, A Man Calle...",1,0.200000,Hybrid
26,A movie about virtual reality,"The Matrix, Ready Player One","The Zero Theorem, Sleep Dealer, eXistenZ, S1m0...",0,0.0,"eXistenZ, Sleep Dealer, S1m0ne, World of Tomor...",1,0.200000,Hybrid


In [20]:
#find where vector search is better
vector_better = comparison_details_df[
    (
        comparison_details_df["vector_rr"]
        >
        comparison_details_df["hybrid_rr"]
    )
]

print(
    "Queries where Vector has better MRR:"
)

print(
    len(vector_better)
)

vector_better

Queries where Vector has better MRR:
1


,question,expected,vector_retrieved,vector_hit,vector_rr,hybrid_retrieved,hybrid_hit,hybrid_rr,winner
25,A movie about aliens,"Arrival, Alien","Encounter, Extinction, Alien Hunter, Lifted, A...",1,0.2,"Aliens in the Attic, AVP: Alien vs. Predator, ...",0,0.0,Vector


In [21]:
#save results
vector_report.to_csv(
    PROJECT_ROOT / "data" / "vector_retrieval_report.csv",
    index=False
)

In [22]:
hybrid_report.to_csv(
    PROJECT_ROOT / "data" / "hybrid_retrieval_report.csv",
    index=False
)

In [23]:
comparison.to_csv(
    PROJECT_ROOT / "data" / "retrieval_method_comparison.csv",
    index=False
)

In [24]:
#conpare detail
comparison_details = []

for i, item in enumerate(ground_truth):

    vector_row = vector_report.iloc[i]
    hybrid_row = hybrid_report.iloc[i]

    comparison_details.append({
        "question": item["question"],
        "expected": ", ".join(item["relevant_movies"]),

        "vector_retrieved": vector_row["retrieved"],
        "vector_hit": vector_row["hit"],
        "vector_rr": vector_row["reciprocal_rank"],

        "hybrid_retrieved": hybrid_row["retrieved"],
        "hybrid_hit": hybrid_row["hit"],
        "hybrid_rr": hybrid_row["reciprocal_rank"]
    })

comparison_details_df = pd.DataFrame(comparison_details)

comparison_details_df

,question,expected,vector_retrieved,vector_hit,vector_rr,hybrid_retrieved,hybrid_hit,hybrid_rr
0,A science fiction movie about space exploration,"Interstellar, The Martian","Red Planet, 3022, Prometheus, Sphere, Life",0,0.0,"Interstellar: Nolan's Odyssey, The Midnight Sk...",0,0.000000
1,A mind-bending science fiction movie,"Inception, Tenet","Transcendence, Bliss, More, Multiverse, Parallel",0,0.0,"Bliss, Transcendence, Metropia, Coherence, Adv...",0,0.000000
2,A movie about time travel,"Back to the Future, Looper","Time Lapse, Timecrimes, Frequently Asked Quest...",0,0.0,"Frequently Asked Questions About Time Travel, ...",0,0.000000
3,A superhero movie with Marvel characters,"Avengers: Endgame, Iron Man","Marvel Studios: Assembling a Universe, Guardia...",0,0.0,"Marvel Studios: Assembling a Universe, Marvel ...",0,0.000000
4,A superhero movie from DC,"The Dark Knight, Man of Steel","DC Showcase: Death, Justice League: Gods and M...",0,0.0,Lego Batman: The Movie - DC Super Heroes Unite...,0,0.000000
5,A fantasy movie with magic,"Harry Potter and the Sorcerer's Stone, Fantast...","Magic Magic, Strange Magic, Bright, Magic in t...",0,0.0,"Onward, Wizards of Waverly Place: The Movie, U...",0,0.000000
6,A movie about dinosaurs,"Jurassic Park, Jurassic World","Dinosaur, The Good Dinosaur, The Dinosaur Proj...",0,0.0,"Walking with Dinosaurs, The Dinosaur Project, ...",0,0.000000
7,A movie about robots and artificial intelligence,"Ex Machina, I, Robot","Automata, I, Robot, A.I. Artificial Intelligen...",1,0.5,"I, Robot, A.I. Artificial Intelligence, Bigbug...",1,1.000000
8,A movie about survival in space,"Gravity, The Martian","3022, Life, Lost in Space, Attraction, Love",0,0.0,"High Life, Passengers, Lost in Space, Prospect...",0,0.000000
9,A movie about dreams,Inception,"The Science of Sleep, The Dreamers, On Body an...",0,0.0,"In My Dreams, In Dreams, A House of Your Dream...",0,0.000000


In [25]:
comparison_details_df.to_csv(
    PROJECT_ROOT / "data" / "retrieval_detailed_comparison.csv",
    index=False
)

In [26]:
#final evaluation statement
if hybrid_metrics["MRR"] > vector_metrics["MRR"]:
    best_method = "Hybrid Search"
elif vector_metrics["MRR"] > hybrid_metrics["MRR"]:
    best_method = "Vector Search"
else:
    best_method = "Tie"

print(f"Best retrieval method based on MRR: {best_method}")

Best retrieval method based on MRR: Hybrid Search


In [27]:
query = "A science fiction movie about space travel"

candidates = hybrid_search_movies(
    query,
    limit=20
)

print(f"Retrieved candidates: {len(candidates)}")

Retrieved candidates: 20


In [29]:
reranked_results = rerank_movies(
    query,
    candidates,
    limit=5
)

AttributeError: 'dict' object has no attribute 'payload'